# Phase 2b — dNBR Label Generation (Burn Scar)

**Environment**: Local  
**Tiempo estimado**: 5–10 min  
**Input**: `data/patches/images/` (5,687 patches, 11 bands)  
**Output**: `data/patches/masks_dnbr/` (binary masks per burn scar)

## Why We Replace FIRMS Labels

The masks in `masks/` use FIRMS detections (active fire): only 148/5,687 patches
have a positive label (2.6%). This provides very few training signals.

**Burn scars** remain visible for weeks after the fire:
- Burned area → NBR drops from ~0.4 → ~-0.2
- dNBR = NBR(pre-fire) − NBR(post-fire) → high values in burned areas
- No external data needed — everything is in the existing patches

## Per-tile Strategy

| Tile  | Available scenes          | Strategy             |
|-------|---------------------------|----------------------|
| 21JVJ | 20211229 (pre) + 20220128 (peak) | **dNBR verdadero** |
| 21JWK | 20211231 (pre) + 20220219 (post) | **dNBR verdadero** |
| 21JUH | 20220131 (peak only)      | NBR threshold        |
| 21JVL | 20220215 (post only)      | NBR threshold        |


In [ ]:
import sys, os
from pathlib import Path

_conda_prefix = Path(sys.executable).parent.parent
_proj_data = _conda_prefix / 'Library' / 'share' / 'proj'
if _proj_data.exists():
    os.environ['PROJ_DATA'] = str(_proj_data)
    os.environ['PROJ_LIB']  = str(_proj_data)

import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

PATCH_DIR    = Path('..') / 'data' / 'patches'
IMG_DIR      = PATCH_DIR / 'images'
MASK_FIRMS   = PATCH_DIR / 'masks'
MASK_DNBR    = PATCH_DIR / 'masks_dnbr'
MASK_DNBR.mkdir(exist_ok=True)

# Índice 0-based en el stack de 11 bandas: [B02,B03,B04,B08,B8A,B11,B12,NDVI,NBR,NDWI,MASK]
NBR_IDX  = 8
MASK_IDX = 10   # 1=píxel válido, 0=nube

print(f'Patches en IMG_DIR: {len(list(IMG_DIR.glob("*.npy")))}')
print(f'Output: {MASK_DNBR.resolve()}')

In [ ]:
# ── Scene classification ───────────────────────────────────────────────────
#
# Stem del patch = stem del GeoTIFF procesado = "{item_id[-15:]}_{date}"
#
# Tile 21JVJ — true dNBR pair
PRE_21JVJ  = '20221230T031220_20211229'
POST_21JVJ = '20240507T114228_20220128'

# Tile 21JWK — true dNBR pair
PRE_21JWK  = '20221228T160256_20211231'
POST_21JWK = '20240516T100304_20220219'

# Map: post scene → pre scene (to recover pre-fire NBR)
DNBR_PAIRS = {
    POST_21JVJ: PRE_21JVJ,
    POST_21JWK: PRE_21JWK,
}

# Pre-fire scenes → all-zeros mask (no scar yet)
PRE_FIRE_STEMS = {PRE_21JVJ, PRE_21JWK}

# Umbrales
DNBR_THRESHOLD = 0.10   # dNBR > 0.10 → cicatriz (baja a alta severidad)
NBR_THRESHOLD  = -0.05  # NBR < -0.05 → píxel quemado (imágenes sin par)

print('Escenas configuradas:')
print(f'  Pre-fuego (zeros): {PRE_FIRE_STEMS}')
print(f'  dNBR verdadero:   {list(DNBR_PAIRS.keys())}')
print(f'  NBR umbral:       21JUH (20220131) + 21JVL (20220215)')

In [ ]:
# ── Pre-load pre-fire scene NBR ───────────────────────────────────────
# For dNBR pairs, we need the pre-fire NBR from the same tile and position.
# Estructura: {stem_pre: {pos_suffix: ndarray(256,256)}}

def get_pos_suffix(path):
    """Extrae '_rXXXXX_cXXXXX' del nombre del patch."""
    stem = Path(path).stem
    idx = stem.find('_r')
    return stem[idx:] if idx != -1 else ''

pre_nbr_cache = {}

print('Pre-loading NBR from reference scenes...')
for pre_stem in PRE_FIRE_STEMS:
    pre_nbr_cache[pre_stem] = {}
    for p in sorted(IMG_DIR.glob(f'{pre_stem}_r*.npy')):
        pos = get_pos_suffix(p)
        img = np.load(p, mmap_mode='r')
        pre_nbr_cache[pre_stem][pos] = img[NBR_IDX].astype(np.float32) / 10000.0
    print(f'  {pre_stem}: {len(pre_nbr_cache[pre_stem])} patches')

In [ ]:
# ── Generate masks_dnbr for all patches ─────────────────────────────────
all_patches = sorted(IMG_DIR.glob('*.npy'))

stats = {'pre_fire': 0, 'dnbr': 0, 'nbr_only': 0}
positive_total = 0
dnbr_values_all = []

for img_path in tqdm(all_patches, desc='Generando máscaras'):
    stem       = img_path.stem
    scene_stem = stem.split('_r')[0]   # part before '_rXXXXX'
    pos        = get_pos_suffix(img_path)
    out_path   = MASK_DNBR / img_path.name

    img   = np.load(img_path)
    nbr   = img[NBR_IDX].astype(np.float32) / 10000.0
    valid = img[MASK_IDX] == 1

    if scene_stem in PRE_FIRE_STEMS:
        burn_mask = np.zeros((256, 256), dtype=np.uint8)
        stats['pre_fire'] += 1

    elif scene_stem in DNBR_PAIRS:
        pre_stem = DNBR_PAIRS[scene_stem]
        if pos in pre_nbr_cache[pre_stem]:
            nbr_pre = pre_nbr_cache[pre_stem][pos]
            dnbr    = nbr_pre - nbr
            burn_mask = ((dnbr > DNBR_THRESHOLD) & valid).astype(np.uint8)
            dnbr_values_all.extend(dnbr[valid].tolist())
        else:
            burn_mask = np.zeros((256, 256), dtype=np.uint8)
        stats['dnbr'] += 1

    else:
        burn_mask = ((nbr < NBR_THRESHOLD) & valid).astype(np.uint8)
        stats['nbr_only'] += 1

    np.save(out_path, burn_mask)
    if burn_mask.max() > 0:
        positive_total += 1

print(f'\n{"─"*50}')
print(f'Patches generated: {len(all_patches)}')
print(f'  Pre-fuego (zeros)   : {stats["pre_fire"]}')
print(f'  dNBR verdadero      : {stats["dnbr"]}')
print(f'  NBR umbral          : {stats["nbr_only"]}')
print(f'\nPatches CON cicatriz: {positive_total}/{len(all_patches)} ({positive_total/len(all_patches)*100:.1f}%)')
print(f'Antes (FIRMS)       : 148/5687 (2.6%)')
print(f'Mejora              : ~{positive_total//148}× more positive examples')

In [ ]:
# ── Distribución de dNBR ──────────────────────────────────────────────────────
if dnbr_values_all:
    dnbr_arr = np.array(dnbr_values_all, dtype=np.float32)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].hist(dnbr_arr, bins=80, range=(-0.5, 1.2),
                 color='steelblue', edgecolor='none', alpha=0.8)
    axes[0].axvline(DNBR_THRESHOLD, color='red', lw=1.5, linestyle='--',
                    label=f'umbral={DNBR_THRESHOLD}')
    axes[0].set_xlabel('dNBR')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('dNBR Distribution — tiles 21JVJ + 21JWK')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    labels = ['Reverdecimiento\n(<-0.10)', 'Sin cambio\n(-0.10–0.10)',
              'Baja sev.\n(0.10–0.27)', 'Mod. baja\n(0.27–0.44)',
              'Mod. alta\n(0.44–0.66)', 'Alta sev.\n(>0.66)']
    bins_sev = [-np.inf, -0.10, 0.10, 0.27, 0.44, 0.66, np.inf]
    counts = [((dnbr_arr >= bins_sev[i]) & (dnbr_arr < bins_sev[i+1])).sum()
              for i in range(len(bins_sev)-1)]
    colors = ['#4CAF50', '#8BC34A', '#FFEB3B', '#FF9800', '#F44336', '#B71C1C']
    bars = axes[1].bar(range(len(labels)), counts, color=colors, edgecolor='white')
    axes[1].set_xticks(range(len(labels)))
    axes[1].set_xticklabels(labels, fontsize=7.5)
    axes[1].set_title('USGS Burn Severity Classification')
    axes[1].set_ylabel('Pixels')
    axes[1].grid(axis='y', alpha=0.3)
    for bar, cnt in zip(bars, counts):
        if sum(counts) > 0:
            axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(counts)*0.01,
                         f'{cnt/sum(counts)*100:.1f}%', ha='center', fontsize=8)

    plt.tight_layout()
    out = Path('..') / 'results' / 'dnbr_distribution.png'
    out.parent.mkdir(exist_ok=True)
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')

In [ ]:
# ── Verificación visual — 3 patches con cicatriz ──────────────────────────────
mask_files     = sorted(MASK_DNBR.glob('*.npy'))
positive_masks = [f for f in mask_files if np.load(f, mmap_mode='r').max() > 0]
print(f'Patches positivos disponibles: {len(positive_masks)}')

if len(positive_masks) >= 3:
    sample_masks = positive_masks[:3]
    fig, axes = plt.subplots(3, 3, figsize=(12, 12))

    for row, mf in enumerate(sample_masks):
        mask  = np.load(mf)
        img   = np.load(IMG_DIR / mf.name)
        firms = (np.load(MASK_FIRMS / mf.name)
                 if (MASK_FIRMS / mf.name).exists() else np.zeros((256,256), dtype=np.uint8))

        def norm(x):
            vals = x[x>0]
            if not len(vals): return np.zeros_like(x, dtype=np.float32)
            lo, hi = np.percentile(vals, [2, 98])
            return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1).astype(np.float32)

        rgb = np.dstack([norm(img[2].astype(np.float32)),
                         norm(img[1].astype(np.float32)),
                         norm(img[0].astype(np.float32))])

        axes[row, 0].imshow(rgb)
        axes[row, 0].set_title(mf.stem[:30], fontsize=7)
        axes[row, 0].axis('off')

        axes[row, 1].imshow(mask, cmap='Reds', vmin=0, vmax=1)
        axes[row, 1].set_title(f'dNBR mask ({mask.mean()*100:.1f}% quemado)', fontsize=9)
        axes[row, 1].axis('off')

        axes[row, 2].imshow(firms, cmap='Oranges', vmin=0, vmax=1)
        axes[row, 2].set_title(f'FIRMS mask ({firms.mean()*100:.1f}% fuego activo)', fontsize=9)
        axes[row, 2].axis('off')

    plt.suptitle('Comparison: dNBR burn scar (new) vs active fire FIRMS (previous)', fontsize=11)
    plt.tight_layout()
    out = Path('..') / 'results' / 'dnbr_vs_firms_comparison.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')
else:
    print(f'Solo {len(positive_masks)} patches positivos.')

---
**Phase 2b complete → copy `masks_dnbr/` to Google Drive, then run `07_prithvi.ipynb` in Colab**

```powershell
copy-item -Recurse .\data\patches\masks_dnbr `"G:\Mon Drive\GeoAI\wildfire-spread\data\patches\masks_dnbr`"
```
